# RQ2 Model Comparison
Kaggle Notebook for RQ2

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:

df = pd.read_csv('/kaggle/input/exam-score-prediction-dataset/exam_score_prediction.csv')
X = df.drop('exam_score', axis=1)
y = df['exam_score']

cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns


In [ ]:

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])


In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

models = {
    "RandomForest": RandomForestRegressor(),
    "SVR": SVR()
}

results = []

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    results.append([name,
                    mean_absolute_error(y_test, preds),
                    mean_squared_error(y_test, preds, squared=False),
                    r2_score(y_test, preds)])

df_res = pd.DataFrame(results, columns=['Model','MAE','RMSE','R2'])
df_res.to_csv('RQ2_table.csv', index=False)

plt.figure()
plt.barh(df_res['Model'], df_res['R2'])
plt.title('RQ2 Model Comparison')
plt.savefig('RQ2_figure.pdf')
plt.close()
